# Colab runner — RoBERTa LOBO sweep

Dartmouth JupyterHub had no GPU, so this runs the GPU work on a Colab **T4**.

**Runtime → Change runtime type → T4 GPU** before you start.

### The one thing that matters here
Free Colab disconnects (~90 min idle, ~4 h max) and wipes the local disk. So results are
written to **Google Drive**, not to the project folder. Every fold that finishes is saved
permanently; re-running a cell after a disconnect skips the folds already done.

Run the stages in order. If you get disconnected, reconnect and re-run from **Stage 4** —
it picks up where it stopped.

## Stage 0 — confirm you actually have a GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
print("device:", torch.cuda.get_device_name(0))

## Stage 1 — mount Drive and choose where results live

Everything written under `WCB_RESULTS_DIR` survives a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE_ROOT = '/content/drive/MyDrive/cs72_final_project'
os.environ['WCB_RESULTS_DIR'] = f'{DRIVE_ROOT}/results'
os.environ['WCB_FIGURES_DIR'] = f'{DRIVE_ROOT}/figures'
for p in (os.environ['WCB_RESULTS_DIR'], os.environ['WCB_FIGURES_DIR']):
    pathlib.Path(p).mkdir(parents=True, exist_ok=True)
print('results ->', os.environ['WCB_RESULTS_DIR'])

## Stage 2 — get the code onto Colab

**Pick ONE.** Option A is better if you have a GitHub repo (and you want one anyway for a
final project). Option B needs no setup: zip the project on your Mac, drop the zip in the
Drive folder above.

In [ ]:
# ---- Option A: clone from GitHub -------------------------------------------------
# !git clone https://github.com/<you>/<repo>.git /content/final_project

# ---- Option B: unzip an archive you put in Drive ----------------------------------
# On your Mac, from the parent of final_project/:
#   zip -r final_project.zip final_project -x '*/.venv/*' '*/__pycache__/*' '*/.git/*'
# then upload final_project.zip into the Drive folder from Stage 1.
!rm -rf /content/final_project
!unzip -q {DRIVE_ROOT}/final_project.zip -d /content/
import os; os.chdir('/content/final_project')
!ls

## Stage 3 — install dependencies

**Do not install torch.** Colab's build is matched to its CUDA driver; replacing it
breaks GPU support. `--no-deps` on the project install stops pip from re-resolving
(and downgrading) what Colab already ships.

The `sys.path` line matters: a running Colab kernel cannot see a package pip installed moments earlier, and restarting the runtime would drop your Drive mount.

In [ ]:
!pip install -q datasets transformers nltk
!pip install -q -e . --no-deps
import nltk; nltk.download('punkt_tab', quiet=True)

import sys, os, pathlib
# pip's editable install writes a .pth file, but site-packages paths are only read at
# interpreter startup -- so this already-running kernel cannot see the new package.
# Adding src/ to sys.path makes it importable without restarting (which would drop the
# Drive mount and the env vars from Stage 1).
root = next((p for p in ('/content/final_project', '/content/final_project/final_project')
             if pathlib.Path(p, 'src', 'wcb', 'config.py').exists()), None)
assert root, "couldn't find src/wcb - check Stage 2 actually unzipped"
os.chdir(root)
if f'{root}/src' not in sys.path:
    sys.path.insert(0, f'{root}/src')

from wcb import config as C
print('project :', root)
print('banks   :', len(C.BANKS))
print('results :', C.RESULTS_DIR)
assert str(C.RESULTS_DIR).startswith('/content/drive'), 'results are NOT on Drive - re-run Stage 1'
print('\nOK - carry on to Stage 4')

## Stage 4 — time one fold

Do this before committing hours. It prints how many folds fit in a session.

Expect roughly **4–10 min/fold** on a T4. Sentences average 34 tokens and the collator
pads per batch rather than to 128, so this usually lands at the fast end.

In [ ]:
!python scripts/run_roberta.py --condition lobo --banks ecb --time-only

## Stage 5 — pick hyperparameters once

Four runs over the WCB paper's grid, selected on the pooled validation split and then
frozen for all 25 folds. Skip this and the defaults (lr 1e-5, batch 32) are used.

In [ ]:
!python scripts/run_roberta.py --hparam-search

## Stage 6 — the sweep, staged so a disconnect still leaves a result

Run these **in order**. Each is independently resumable; the ordering means that even if
you lose the session partway you have something coherent to write up.

| cell | folds | time | what you have afterwards |
|---|---|---|---|
| 6a | 2 | ~10 min | ceiling + size control for all 25 banks |
| 6b | 5 | ~25 min | a complete story for the 5 focus banks |
| 6c | 20 | ~2–3 h | the full 25-bank sweep |

In [ ]:
# 6a - pooled + size control (also saves the checkpoint the demo and Czech check need)
!python scripts/run_roberta.py --condition pooled size_control

In [ ]:
# 6b - the 5 focus banks: Turkey, China, Brazil, ECB, Poland
!python scripts/run_roberta.py --condition lobo --banks focus

In [ ]:
# 6c - the remaining 20 banks. Re-run this after any disconnect; it skips what is done.
!python scripts/run_roberta.py --condition lobo

## Stage 7 — verify against the two published numbers

Pooled weighted F1 should land near **0.714** and the Czech hold-out near **0.800**, both
from the WCB paper. **If these are off, stop and debug before trusting the LOBO numbers** —
it means preprocessing or label mapping is wrong.

In [ ]:
CKPT = f"{C.RESULTS_DIR}/roberta/seed5768/pooled_model"
!python scripts/verify.py --model roberta --checkpoint {CKPT}

## Stage 8 — tables and figures

In [ ]:
!python scripts/make_report.py --model roberta

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f"{C.FIGURES_DIR}/roberta_fig*.png")):
    print(f); display(Image(f))

### Bring the baseline results across

The TF-IDF results were produced on your laptop and live in the repo, while Colab writes
to Drive. Copying them over lets `make_report.py` run for both models in one place, so
the writeup can put RoBERTa and the baseline side by side.

In [ ]:
import shutil, pathlib
src = pathlib.Path('/content/final_project/results/baseline')
dst = pathlib.Path(C.RESULTS_DIR) / 'baseline'
if src.exists() and not dst.exists():
    shutil.copytree(src, dst)
    print('copied', len(list(dst.rglob('*.json'))), 'baseline fold results to Drive')
else:
    print('already present' if dst.exists() else 'no baseline results in the archive')

!python scripts/make_report.py --model baseline

## Stage 9 — how much progress have I made?

Safe to run at any time, including after a disconnect.

In [ ]:
import glob, os
for cond in ('pooled', 'size_control', 'lobo'):
    done = glob.glob(f"{C.RESULTS_DIR}/roberta/seed5768/{cond}/*.json")
    print(f"{cond:<14} {len(done):>2}/{len(C.BANKS) if cond=='lobo' else len(C.BANKS)} banks scored")
missing = [b for b in C.BANKS
           if not os.path.exists(f"{C.RESULTS_DIR}/roberta/seed5768/lobo/{b}.json")]
print("\nLOBO folds still to run:", missing or "none - sweep complete")